# Project 93 — Local PR Review Assistant
## Summarize Code Diffs and Suggest Issues

**Stack:** Ollama · LangChain · Git Diff Parsing · Jupyter

## Project Overview

This notebook builds a **local PR review assistant** that parses unified git diffs,
analyzes code changes, and produces structured review feedback — all powered by a
local LLM via Ollama.

Everything runs **locally** — no code leaves your machine.

### What You Will Learn

1. How to **parse unified git diffs** into structured per-file change data
2. How to build LLM chains that produce **code review feedback**
3. How to perform **security-focused review** of code changes
4. How to assess **risk level** of pull requests
5. How to generate a **team-facing review summary**
6. Practical prompt-engineering patterns for code-review automation

## Prerequisites

| Requirement | Details |
|---|---|
| **Ollama** | Running locally with `qwen3:8b` pulled |
| **Python packages** | `langchain`, `langchain-ollama` |

```bash
# Pull the model (run in terminal)
ollama pull qwen3:8b
```

In [ ]:
# !pip install -q langchain langchain-ollama

## Step 1 — Imports and Configuration

In [ ]:
import json
import re
import textwrap

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Configuration ──────────────────────────────────────────────
OLLAMA_MODEL = "qwen3:8b"
TEMPERATURE = 0.1

print("Configuration ready.")

## Step 2 — Initialize LLM

We use **Qwen3 8B** via Ollama with a low temperature for precise,
code-focused analysis.

In [ ]:
llm = ChatOllama(model=OLLAMA_MODEL, temperature=TEMPERATURE)

# Connectivity check
test_msg = llm.invoke("Reply with only: OK")
print(f"LLM ready: {test_msg.content.strip()[:20]}")

## Step 3 — Define Sample Pull Requests

We define three realistic pull requests with unified git diffs. Each PR
represents a different type of change:

| PR | Type | Key Concern |
|---|---|---|
| PR-142 | New feature — rate limiting | In-memory state, thread safety |
| PR-143 | Security fix — SQL injection | Parameterized queries |
| PR-144 | Schema change — new table | Migration completeness, missing columns |

In [ ]:
PULL_REQUESTS = [
    {
        "id": "PR-142",
        "title": "Add rate limiting to API endpoints",
        "author": "alice",
        "files_changed": 2,
        "diff": textwrap.dedent("""\
            --- a/api/middleware.py
            +++ b/api/middleware.py
            @@ -1,5 +1,28 @@
            +from time import time
            +from collections import defaultdict
            +
            +class RateLimiter:
            +    def __init__(self, max_requests=100, window=60):
            +        self.max_requests = max_requests
            +        self.window = window
            +        self.requests = defaultdict(list)
            +
            +    def is_allowed(self, client_ip):
            +        now = time()
            +        self.requests[client_ip] = [
            +            t for t in self.requests[client_ip] if now - t < self.window
            +        ]
            +        if len(self.requests[client_ip]) >= self.max_requests:
            +            return False
            +        self.requests[client_ip].append(now)
            +        return True
            +
             from flask import Flask, request, jsonify
            +
            +limiter = RateLimiter()
            +
             @app.before_request
             def check_rate_limit():
            +    if not limiter.is_allowed(request.remote_addr):
            +        return jsonify({"error": "rate limited"}), 429
            """)
    },
    {
        "id": "PR-143",
        "title": "Fix SQL injection in user search",
        "author": "bob",
        "files_changed": 1,
        "diff": textwrap.dedent("""\
            --- a/api/users.py
            +++ b/api/users.py
            @@ -10,7 +10,8 @@
             def search_users(query):
            -    sql = f"SELECT * FROM users WHERE name LIKE '%{query}%'"
            -    results = db.execute(sql)
            +    sql = "SELECT * FROM users WHERE name LIKE :query"
            +    results = db.execute(sql, {"query": f"%{query}%"})
                 return results
            """)
    },
    {
        "id": "PR-144",
        "title": "Add user preferences table",
        "author": "carol",
        "files_changed": 2,
        "diff": textwrap.dedent("""\
            --- a/models/user.py
            +++ b/models/user.py
            @@ -15,6 +15,20 @@
            +class UserPreference(Base):
            +    __tablename__ = 'user_preferences'
            +    id = Column(Integer, primary_key=True)
            +    user_id = Column(Integer, ForeignKey('users.id'))
            +    key = Column(String(100))
            +    value = Column(Text)
            +    created_at = Column(DateTime, default=datetime.utcnow)
            +
            --- /dev/null
            +++ b/migrations/add_preferences.py
            @@ -0,0 +1,8 @@
            +def upgrade():
            +    op.create_table('user_preferences',
            +        sa.Column('id', sa.Integer, primary_key=True),
            +        sa.Column('user_id', sa.Integer, sa.ForeignKey('users.id')),
            +        sa.Column('key', sa.String(100)),
            +        sa.Column('value', sa.Text),
            +    )
            """)
    },
]

print(f"Pull requests to review: {len(PULL_REQUESTS)}")
for pr in PULL_REQUESTS:
    print(f"  {pr['id']}: {pr['title']} (by {pr['author']}, {pr['files_changed']} files)")

## Step 4 — Diff Parsing Utility

Before sending diffs to the LLM, we parse them into structured data:
- **File path** (old and new)
- **Lines added** and **lines removed**
- **Change type** (modified, added, deleted)

This lets us give the LLM richer context and also produce quantitative stats.

In [ ]:
def parse_unified_diff(diff_text: str) -> list[dict]:
    """Parse a unified diff into a list of per-file change records.

    Each record has: old_file, new_file, change_type, additions, deletions, hunks.
    """
    files = []
    current = None
    hunk_lines = []

    for line in diff_text.splitlines():
        stripped = line.strip()

        if stripped.startswith("--- "):
            # Save previous file record
            if current is not None:
                current["hunks"] = "\n".join(hunk_lines)
                files.append(current)
            old_file = stripped[4:].strip()
            current = {"old_file": old_file, "new_file": "", "change_type": "modified",
                       "additions": 0, "deletions": 0, "hunks": ""}
            hunk_lines = []
            if old_file == "/dev/null":
                current["change_type"] = "added"

        elif stripped.startswith("+++ ") and current is not None:
            new_file = stripped[4:].strip()
            current["new_file"] = new_file
            if new_file == "/dev/null":
                current["change_type"] = "deleted"

        elif current is not None:
            hunk_lines.append(stripped)
            if stripped.startswith("+") and not stripped.startswith("+++"):
                current["additions"] += 1
            elif stripped.startswith("-") and not stripped.startswith("---"):
                current["deletions"] += 1

    # Save last file record
    if current is not None:
        current["hunks"] = "\n".join(hunk_lines)
        files.append(current)

    return files


# Parse all PRs
for pr in PULL_REQUESTS:
    pr["parsed_files"] = parse_unified_diff(pr["diff"])

# Show parsed results
for pr in PULL_REQUESTS:
    print(f"\n{pr['id']}: {pr['title']}")
    for f in pr["parsed_files"]:
        path = f["new_file"] if f["new_file"] != "/dev/null" else f["old_file"]
        print(f"  {f['change_type']:>8}  +{f['additions']} -{f['deletions']}  {path}")

## Step 5 — Build the Code Review Chain

The review prompt instructs the LLM to act as a senior code reviewer.
It analyzes the diff and produces:

- A **summary** of what the PR does
- **Issues found** (bugs, style, security, design)
- A **risk level** (low / medium / high / critical)
- A **recommendation** (approve / request changes / comment)

In [ ]:
REVIEW_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a senior software engineer performing a code review.
Analyze the pull request diff and provide a structured review.

Your review must include:
1. SUMMARY: One paragraph describing what the PR does.
2. ISSUES: A numbered list of issues found. For each issue state:
   - Severity (critical / warning / suggestion)
   - File and context
   - Description of the problem
   - Suggested fix (if applicable)
3. POSITIVES: What the PR does well.
4. RISK LEVEL: low / medium / high / critical — with justification.
5. RECOMMENDATION: approve / request_changes / comment.

Be specific — reference file names and code patterns from the diff."""),
    ("human", """Review this pull request:

PR: {pr_id} — {title}
Author: {author}
Files changed: {files_changed}

Diff:
{diff}""")
])

review_chain = REVIEW_PROMPT | llm | StrOutputParser()

print("Code review chain ready.")

## Step 6 — Review PR-142: Rate Limiting

This PR adds a new `RateLimiter` class and wires it into Flask middleware.
Key concerns: in-memory state, thread safety, no persistence.

In [ ]:
pr = PULL_REQUESTS[0]
review_142 = review_chain.invoke({
    "pr_id": pr["id"],
    "title": pr["title"],
    "author": pr["author"],
    "files_changed": pr["files_changed"],
    "diff": pr["diff"],
})

print(f"REVIEW — {pr['id']}: {pr['title']}")
print("=" * 60)
print(review_142)

## Step 7 — Review PR-143: SQL Injection Fix

This PR fixes a critical SQL injection vulnerability by switching
from f-string interpolation to parameterized queries.

In [ ]:
pr = PULL_REQUESTS[1]
review_143 = review_chain.invoke({
    "pr_id": pr["id"],
    "title": pr["title"],
    "author": pr["author"],
    "files_changed": pr["files_changed"],
    "diff": pr["diff"],
})

print(f"REVIEW — {pr['id']}: {pr['title']}")
print("=" * 60)
print(review_143)

## Step 8 — Review PR-144: User Preferences Table

This PR adds a new database model and migration. Key concerns:
migration completeness, missing `created_at` in migration, index coverage.

In [ ]:
pr = PULL_REQUESTS[2]
review_144 = review_chain.invoke({
    "pr_id": pr["id"],
    "title": pr["title"],
    "author": pr["author"],
    "files_changed": pr["files_changed"],
    "diff": pr["diff"],
})

print(f"REVIEW — {pr['id']}: {pr['title']}")
print("=" * 60)
print(review_144)

## Step 9 — Security-Focused Review

We run a **second pass** focused specifically on security concerns.
This catches vulnerabilities that a general review might miss:
injection, auth bypass, data exposure, etc.

In [ ]:
SECURITY_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a security engineer reviewing code changes.
Focus ONLY on security concerns:

- Injection vulnerabilities (SQL, command, XSS)
- Authentication / authorization issues
- Data exposure or leakage
- Insecure defaults or configurations
- Missing input validation
- Race conditions or state issues
- Denial of service vectors

For each finding, state:
  SEVERITY: critical / high / medium / low
  FINDING: What the issue is
  IMPACT: What could go wrong
  FIX: How to resolve it

If no security issues are found, say so explicitly."""),
    ("human", "Security review for {pr_id} — {title}:\n\n{diff}")
])

security_chain = SECURITY_PROMPT | llm | StrOutputParser()

all_reviews = [review_142, review_143, review_144]
security_reviews = []
for pr in PULL_REQUESTS:
    sec = security_chain.invoke({
        "pr_id": pr["id"],
        "title": pr["title"],
        "diff": pr["diff"],
    })
    security_reviews.append(sec)
    print(f"\nSECURITY REVIEW — {pr['id']}")
    print("-" * 50)
    print(sec[:600])
    if len(sec) > 600:
        print(f"... ({len(sec)} chars total)")
    print()

## Step 10 — Risk Assessment

We ask the LLM to produce a concise **risk classification** for each PR.
This can be used to prioritize reviews or gate deployments.

In [ ]:
RISK_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Classify the risk level of this pull request.

Return EXACTLY this format (one line each):
RISK: <low|medium|high|critical>
DECISION: <approve|request_changes|comment>
BREAKING: <yes|no>
REASON: <one sentence justification>"""),
    ("human", "PR: {pr_id} — {title}\nDiff:\n{diff}")
])

risk_chain = RISK_PROMPT | llm | StrOutputParser()

risk_results = []
for pr in PULL_REQUESTS:
    risk_raw = risk_chain.invoke({
        "pr_id": pr["id"],
        "title": pr["title"],
        "diff": pr["diff"],
    })
    risk_results.append({"pr_id": pr["id"], "title": pr["title"],
                         "author": pr["author"], "raw": risk_raw})
    print(f"{pr['id']}: {risk_raw.strip()[:200]}")
    print()

## Step 11 — Review Dashboard

We compile all reviews into a summary table showing the key metrics
for each PR at a glance.

In [ ]:
def extract_field(text: str, field: str) -> str:
    """Extract a field value from structured LLM output."""
    pattern = rf"{field}:\s*(.+)"
    match = re.search(pattern, text, re.IGNORECASE)
    return match.group(1).strip() if match else "unknown"


print("PR REVIEW DASHBOARD")
print("=" * 70)
print(f"{'PR':<10} {'Title':<35} {'Risk':<10} {'Decision':<16} {'Author'}")
print("-" * 70)

for r in risk_results:
    risk = extract_field(r["raw"], "RISK")
    decision = extract_field(r["raw"], "DECISION")
    print(f"{r['pr_id']:<10} {r['title'][:33]:<35} {risk:<10} {decision:<16} {r['author']}")

print("-" * 70)

# Diff stats
print("\nDIFF STATISTICS")
print(f"{'PR':<10} {'Files':<8} {'Added':<8} {'Removed':<8}")
print("-" * 40)
for pr in PULL_REQUESTS:
    adds = sum(f["additions"] for f in pr["parsed_files"])
    dels = sum(f["deletions"] for f in pr["parsed_files"])
    print(f"{pr['id']:<10} {len(pr['parsed_files']):<8} +{adds:<7} -{dels:<7}")

## Step 12 — Team Summary Generation

Finally, we generate a **team-facing summary** of all PR reviews.
This is the kind of digest a tech lead would share in a team channel.

In [ ]:
SUMMARY_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Write a concise team-facing summary of today's PR reviews.
Highlight:
- Security fixes that should be fast-tracked
- Breaking changes or risky PRs that need extra attention
- Items that are safe to merge

Keep it under 300 words. Use a professional but friendly tone."""),
    ("human", "PR reviews to summarize:\n\n{reviews}")
])

summary_chain = SUMMARY_PROMPT | llm | StrOutputParser()

# Combine all review text
combined = ""
for i, pr in enumerate(PULL_REQUESTS):
    combined += f"\n--- {pr['id']}: {pr['title']} ---\n"
    combined += all_reviews[i][:500] + "\n"
    combined += f"Risk assessment: {risk_results[i]['raw'][:200]}\n"

team_summary = summary_chain.invoke({"reviews": combined})

print("TEAM SUMMARY")
print("=" * 60)
print(team_summary)

## Step 13 — Limitations and Failure Cases

Let's test the reviewer on a **tricky diff** that contains a subtle bug
to see if the LLM catches it.

In [ ]:
TRICKY_DIFF = textwrap.dedent("""\
    --- a/auth/login.py
    +++ b/auth/login.py
    @@ -20,8 +20,10 @@
     def authenticate(username, password):
         user = db.get_user(username)
    -    if user and user.check_password(password):
    +    if user.check_password(password):
             return create_token(user)
    -    return None
    +    # Allow guest access for demo mode
    +    if os.environ.get("DEMO_MODE"):
    +        return create_token(GuestUser())
    +    return None
""")

tricky_review = review_chain.invoke({
    "pr_id": "PR-999",
    "title": "Add demo mode guest access",
    "author": "intern",
    "files_changed": 1,
    "diff": TRICKY_DIFF,
})

print("TRICKY REVIEW — PR-999: Add demo mode guest access")
print("=" * 60)
print(tricky_review)
print("\n--- Did the reviewer catch these issues? ---")
print("1. Removed null check on 'user' → AttributeError if user not found")
print("2. DEMO_MODE env var could be set in production → auth bypass")
print("3. GuestUser() token has unknown permissions scope")

## Evaluation Summary

| Dimension | How We Evaluated |
|---|---|
| **Diff parsing** | Verified `parse_unified_diff()` extracts correct file counts, additions, deletions |
| **Review quality** | Inspected whether reviews mention file-specific issues from the diff |
| **Security analysis** | Checked whether the security pass catches known vulnerabilities |
| **Risk assessment** | Verified risk levels are reasonable for each PR type |
| **Edge-case detection** | Tested a tricky diff with subtle bugs (Step 13) |
| **Summary quality** | Checked team summary highlights key action items |

### Known Limitations

- **Context window**: The LLM only sees the diff, not the full codebase.
  It cannot verify whether imports exist, tests pass, or types are correct.
- **False positives**: The reviewer may flag non-issues, especially in
  unfamiliar frameworks or domain-specific code.
- **Structured output**: Without JSON mode, the LLM's risk/decision fields
  may not always follow the exact requested format.
- **Large diffs**: Very large PRs (>500 lines) may exceed the model's
  effective context and produce shallow reviews.
- **No test verification**: The reviewer can suggest tests are needed but
  cannot verify whether tests already exist in the repo.

## How to Improve This Project

1. **Real git integration** — use `subprocess` to run `git diff` on actual repos
2. **Multi-file context** — fetch full file content for referenced imports/classes
3. **Iterative review** — if risk is high, run a second deeper analysis pass
4. **Structured output** — use JSON mode or Pydantic with `with_structured_output()`
   for machine-parseable reviews
5. **Review history** — track review patterns per author to identify coaching areas
6. **CI integration** — trigger reviews automatically on PR creation via webhooks

## What You Learned

- **Diff parsing** — extracting structured data from unified git diffs
- **Code review automation** — using LLMs to analyze code changes
- **Security analysis** — running a focused security review pass
- **Risk assessment** — classifying PR risk for deployment decisions
- **Team summaries** — generating human-readable review digests
- **Edge-case testing** — evaluating reviewer quality on tricky diffs